# Step 7 — ERFNet Pixel-Based Anomaly Baselines
**Methods:** MSP · Max Logit · Max Entropy  
**Datasets:** SMIYC RA-21, SMIYC RO-21, FS Lost&Found, FS Static, Road Anomaly

---

## How to Run

1. **GPU:** Settings → Accelerator → GPU T4 x2 (or P100)
2. **Dataset:** Upload `Anomaly_Validation_Datasets.zip` as a Kaggle Dataset, then add it via *Add Input*
3. **Path:** Update `ANOMALY_ROOT` in Cell 1 to match your dataset's path under `/kaggle/input/`
4. Run all cells top-to-bottom — results are saved to `/kaggle/working/results/erfnet/`


In [ ]:
import torch, os
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

## 1 — Paths Setup

Adjust the paths to your Kaggle datasets. Edit `REPO_ROOT` and `ANOMALY_ROOT` based on how you uploaded the data.

In [ ]:
import os, sys, glob
from pathlib import Path

# ═══════════════════════════════════════════════════════════════
#  1. CLONE THE PROFESSOR'S REPO (ERFNet + weights)
# ═══════════════════════════════════════════════════════════════

REPO_ROOT = '/kaggle/working/AnomalySegmentation_CourseProjectBaseCode'

if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/shyam671/AnomalySegmentation_CourseProjectBaseCode.git {REPO_ROOT}
    print('Repo cloned!')
else:
    print('Repo already present.')

# ═══════════════════════════════════════════════════════════════
#  2. ANOMALY DATASETS PATH (uploaded as Kaggle Dataset)
# ═══════════════════════════════════════════════════════════════
#  Upload Anomaly_Validation_Datasets.zip as a Kaggle Dataset:
#    - kaggle.com > Your Work > Datasets > New Dataset > Upload
#    - Then in the notebook: Add Input > search for the dataset name
#  The exact path is under /kaggle/input/<dataset-name>/

ANOMALY_ROOT = "/kaggle/input/datasets/federicoremy/anomaly-validation/Validation_Dataset"
# this is the path I got when I uploaded the dataset, but it may be different for you, so check carefully!

# Working directory
RESULTS_DIR = '/kaggle/working/results/erfnet'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════════
#  VERIFY
# ═══════════════════════════════════════════════════════════════
print('Repo exists:', os.path.exists(REPO_ROOT))
print('Anomaly data exists:', os.path.exists(ANOMALY_ROOT))

# If the path does not exist, search automatically
if not os.path.exists(ANOMALY_ROOT):
    print('\nANOMALY_ROOT not found! Searching...')
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'RoadAnomaly21' in dirs or 'fs_static' in dirs:
            print(f'  FOUND: {root}')
            print(f'  -> Update ANOMALY_ROOT = "{root}"')
            break
        # search also a level deeper
        for d in dirs:
            subpath = os.path.join(root, d)
            if os.path.isdir(subpath):
                subdirs = os.listdir(subpath)
                if 'RoadAnomaly21' in subdirs or 'fs_static' in subdirs:
                    print(f'  FOUND: {subpath}')
                    print(f'  -> Update ANOMALY_ROOT = "{subpath}"')
                    break
else:
    print('\nAnomaly dataset contents:')
    for item in sorted(os.listdir(ANOMALY_ROOT)):
        full = os.path.join(ANOMALY_ROOT, item)
        if os.path.isdir(full):
            n = len(os.listdir(full))
            print(f'  {item}/ ({n} items)')

if os.path.exists(REPO_ROOT):
    print('\nRepo contents:')
    for item in sorted(os.listdir(REPO_ROOT)):
        print(f'  {item}')


## 2 — Inspect Dataset Structure

This is **essential**: understand how images and labels are organised for each dataset before proceeding.

In [ ]:
# explores the directory tree up to a certain depth, showing file sizes and folder item counts  
import os

def show_tree(path, prefix="", max_depth=3, current_depth=0):
    if current_depth >= max_depth:
        return
    if not os.path.exists(path):
        print(f"{prefix}[NOT FOUND] {path}")
        return
    items = sorted(os.listdir(path))
    for i, item in enumerate(items[:15]):  # limit display
        full = os.path.join(path, item)
        connector = "├── " if i < len(items) - 1 else "└── "
        if os.path.isdir(full):
            n = len([f for f in os.listdir(full) if not f.startswith('.')])
            print(f"{prefix}{connector}📁 {item}/ ({n} items)")
            show_tree(full, prefix + ("│   " if i < len(items) - 1 else "    "), max_depth, current_depth + 1)
        else:
            size_kb = os.path.getsize(full) / 1024
            print(f"{prefix}{connector}📄 {item} ({size_kb:.0f} KB)")
    if len(items) > 15:
        print(f"{prefix}... and {len(items) - 15} more items")

print("=" * 60)
print("ANOMALY DATASET STRUCTURE")
print("=" * 60)
show_tree(ANOMALY_ROOT, max_depth=3)


## 3 — Dataset Configuration

After inspecting the structure (Cell 2), configure the paths for each dataset here.

**Standard SMIYC / FishyScapes label convention:**
- `0` = in-distribution (road, known classes)
- `1` = anomaly (OOD)
- `255` = void / ignore

**Edit the paths below** based on the output of Cell 2!

In [ ]:
# DATASET CONFIGURATION — based on the actual directory structure

import glob

def find_images(folder, exts=('png', 'jpg', 'jpeg', 'webp')):
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(folder, f'*.{ext}')))
    return sorted(files)

DATASETS = {
    "SMIYC_RA21": {
        "image_dir": f"{ANOMALY_ROOT}/RoadAnomaly21/images",
        "label_dir": f"{ANOMALY_ROOT}/RoadAnomaly21/labels_masks",
        "display_name": "SMIYC RA-21",
    },
    "SMIYC_RO21": {
        "image_dir": f"{ANOMALY_ROOT}/RoadObsticle21/images",
        "label_dir": f"{ANOMALY_ROOT}/RoadObsticle21/labels_masks",
        "display_name": "SMIYC RO-21",
    },
    "FS_LostFound": {
        "image_dir": f"{ANOMALY_ROOT}/FS_LostFound_full/images",
        "label_dir": f"{ANOMALY_ROOT}/FS_LostFound_full/labels_masks",
        "display_name": "FS L&F",
    },
    "FS_Static": {
        "image_dir": f"{ANOMALY_ROOT}/fs_static/images",
        "label_dir": f"{ANOMALY_ROOT}/fs_static/labels_masks",
        "display_name": "FS Static",
    },
    "RoadAnomaly": {
        "image_dir": f"{ANOMALY_ROOT}/RoadAnomaly/images",
        "label_dir": f"{ANOMALY_ROOT}/RoadAnomaly/labels_masks",
        "display_name": "Road Anomaly",
    },
}

# Verify dataset paths and counts
print("Dataset file counts:")
print("-" * 60)
for key, cfg in DATASETS.items():
    imgs = find_images(cfg["image_dir"])
    lbls = find_images(cfg["label_dir"])
    ok = len(imgs) > 0 and len(lbls) > 0
    mark = "OK" if ok else "MISSING"
    print(f"  [{mark}] {cfg['display_name']:15s} -> {len(imgs):3d} images, {len(lbls):3d} labels")
    if len(imgs) == 0:
        print(f"     Check: {cfg['image_dir']}")
        parent = os.path.dirname(cfg['image_dir'])
        if os.path.exists(parent):
            print(f"     Parent contents: {os.listdir(parent)}")
    if len(imgs) > 0:
        exts = set(os.path.splitext(f)[1] for f in imgs)
        print(f"     Extensions: {exts}")


## 4 — Load ERFNet Model and Weights

In [ ]:
import sys, shutil

# Repo already in /kaggle/working/ (cloned in cell 1)
# Add directories to the Python path
sys.path.insert(0, os.path.join(REPO_ROOT, 'train'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'eval'))

# Search for ERFNet weights
import glob
weight_candidates = (
    glob.glob(f'{REPO_ROOT}/trained_models/*.pth') +
    glob.glob(f'{REPO_ROOT}/trained_models/*.pth.tar') +
    glob.glob(f'{REPO_ROOT}/**/*erfnet*.pth', recursive=True)
)
print('Weight candidates:', weight_candidates)

# Also show trained_models/ contents
tm_dir = os.path.join(REPO_ROOT, 'trained_models')
if os.path.exists(tm_dir):
    print('\ntrained_models/ contents:')
    for f in os.listdir(tm_dir):
        size_mb = os.path.getsize(os.path.join(tm_dir, f)) / 1e6
        print(f'  {f} ({size_mb:.1f} MB)')


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms

# Import ERFNet
try:
    from erfnet import ERFNet
    print('Loaded ERFNet from erfnet module')
except ImportError:
    try:
        from models.erfnet import ERFNet
        print('Loaded ERFNet from models.erfnet')
    except ImportError:
        print('Could not import ERFNet!')
        raise

# The checkpoint uses 20 classes (19 Cityscapes + 1 void/unlabeled)
NUM_CLASSES = 20

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

ERF_WEIGHTS = weight_candidates[0] if weight_candidates else None
assert ERF_WEIGHTS is not None, 'Pesi ERFNet non trovati!'

model = ERFNet(NUM_CLASSES)
ckpt = torch.load(ERF_WEIGHTS, map_location='cpu', weights_only=False)

if isinstance(ckpt, dict) and 'state_dict' in ckpt:
    state_dict = ckpt['state_dict']
else:
    state_dict = ckpt

# Strip 'module.' prefix (from DataParallel)
cleaned = {}
for k, v in state_dict.items():
    new_k = k.replace('module.', '')
    cleaned[new_k] = v

missing, unexpected = model.load_state_dict(cleaned, strict=False)
if missing:
    print(f'Missing keys (ok se solo encoder.output_conv): {missing}')
if unexpected:
    print(f'Unexpected keys: {unexpected}')

model.eval().to(device)
print(f'ERFNet loaded from: {ERF_WEIGHTS}')
print(f'Output classes: {NUM_CLASSES}')

## 5 — Post-Hoc Scoring Methods

Three required methods:

1. **MSP** (Maximum Softmax Probability): `anomaly_score = 1 - max(softmax(logits))`  
   → Low confidence = high anomaly score

2. **Max Logit**: `anomaly_score = -max(logits)`  
   → Low max logit = high anomaly score (negated so higher = more anomalous)

3. **Max Entropy**: `anomaly_score = -Σ p·log(p)`  
   → High entropy = high uncertainty = more anomalous

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  POST-HOC ANOMALY SCORING METHODS
# ═══════════════════════════════════════════════════════════════

def compute_msp(logits):
    """
    Maximum Softmax Probability.
    Anomaly score = 1 - max_c softmax(logits)_c
    Higher score → more likely anomalous.
    """
    probs = F.softmax(logits, dim=1)             # (B, C, H, W)
    max_prob, _ = torch.max(probs, dim=1)         # (B, H, W)
    return 1.0 - max_prob                          # (B, H, W)


def compute_max_logit(logits):
    """
    Max Logit (negated).
    Anomaly score = -max_c logits_c
    A confident in-distribution pixel has high max logit → low anomaly score.
    """
    max_logit, _ = torch.max(logits, dim=1)       # (B, H, W)
    return -max_logit                              # (B, H, W)


def compute_max_entropy(logits):
    """
    Predictive Entropy.
    anomaly_score = -Σ_c p_c · log(p_c)
    Uniform distribution → max entropy → most anomalous.
    """
    probs = F.softmax(logits, dim=1)              # (B, C, H, W)
    log_probs = torch.log(probs + 1e-10)
    entropy = -torch.sum(probs * log_probs, dim=1) # (B, H, W)
    return entropy                                  # (B, H, W)


METHODS = {
    "MSP":         compute_msp,
    "MaxLogit":    compute_max_logit,
    "MaxEntropy":  compute_max_entropy,
}

print("✅ Defined 3 anomaly scoring methods:", list(METHODS.keys()))


## 6 — Evaluation Metrics: AuPRC and FPR@95TPR

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve
import numpy as np

def compute_auprc(anomaly_scores, ood_labels):
    """
    Area Under the Precision-Recall Curve.
    anomaly_scores: flat array, higher = more anomalous
    ood_labels: flat binary array, 1 = OOD, 0 = in-distribution
    """
    return average_precision_score(ood_labels, anomaly_scores)


def compute_fpr_at_95_tpr(anomaly_scores, ood_labels):
    """
    False Positive Rate at 95% True Positive Rate.
    """
    pos = ood_labels == 1
    neg = ood_labels == 0
    n_pos = pos.sum()
    n_neg = neg.sum()

    if n_pos == 0 or n_neg == 0:
        return float('nan')

    # Sort by descending anomaly score
    sorted_idx = np.argsort(-anomaly_scores)
    sorted_labels = ood_labels[sorted_idx]

    # Walk through sorted predictions, count TPs and FPs
    tp_cumsum = np.cumsum(sorted_labels == 1)
    fp_cumsum = np.cumsum(sorted_labels == 0)

    tpr = tp_cumsum / n_pos
    fpr = fp_cumsum / n_neg

    # Find the first index where TPR >= 0.95
    idx = np.searchsorted(tpr, 0.95)

    if idx >= len(fpr):
        return 1.0

    return fpr[idx]

print("✅ Metric functions defined: AuPRC, FPR@95TPR")


## 7 — Image and Label Preprocessing

**Important:** the image transform includes ImageNet normalisation,  
but labels must only be resized (nearest neighbour) and converted to numpy arrays.

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((512, 1024), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def load_and_preprocess_image(img_path):
    img = Image.open(img_path).convert("RGB")
    return image_transform(img).unsqueeze(0)

def load_label(label_path, target_size=(512, 1024)):
    label = Image.open(label_path)
    label = label.resize((target_size[1], target_size[0]),
                         resample=Image.NEAREST)
    return np.array(label, dtype=np.uint8)

# Quick test
for ds_key, ds_cfg in DATASETS.items():
    imgs = find_images(ds_cfg["image_dir"])
    lbls = find_images(ds_cfg["label_dir"])
    if imgs and lbls:
        test_img = load_and_preprocess_image(imgs[0])
        test_lbl = load_label(lbls[0])
        print(f"Test {ds_cfg['display_name']}:")
        print(f"  Image tensor shape: {test_img.shape}")
        print(f"  Label shape: {test_lbl.shape}")
        print(f"  Label unique values: {np.unique(test_lbl)}")
        break

## 8 — Main Evaluation Loop

Evaluates each method on each dataset separately, as required by the project table.

In [ ]:
!pip install -q scikit-learn

In [ ]:
import time
from collections import defaultdict

def get_paired_files(image_dir, label_dir):
    """Pair images and labels by stem (filename without extension)."""
    imgs = find_images(image_dir)
    lbls = find_images(label_dir)
    
    img_by_stem = {Path(p).stem: p for p in imgs}
    lbl_by_stem = {Path(p).stem: p for p in lbls}
    
    common = sorted(set(img_by_stem) & set(lbl_by_stem))
    
    if len(common) == 0:
        print(f"    No stem matches! Trying sorted-order pairing...")
        print(f"       Sample img stems: {list(img_by_stem.keys())[:3]}")
        print(f"       Sample lbl stems: {list(lbl_by_stem.keys())[:3]}")
        return list(zip(sorted(imgs), sorted(lbls)))
    
    return [(img_by_stem[s], lbl_by_stem[s]) for s in common]


def evaluate_dataset(ds_key, ds_cfg, model, methods, device):
    """Evaluate all methods on a single dataset."""
    paired = get_paired_files(ds_cfg["image_dir"], ds_cfg["label_dir"])
    assert len(paired) > 0, f'No paired images/labels for {ds_key}'
    
    all_scores = {m: [] for m in methods}
    all_labels = {m: [] for m in methods}
    
    print(f"  Processing {len(paired)} image-label pairs...")
    
    with torch.no_grad():
        for i, (img_path, lbl_path) in enumerate(paired):
            img_tensor = load_and_preprocess_image(img_path).to(device)
            label = load_label(lbl_path)
            
            # Road Anomaly uses value 2 for anomaly pixels
            if ds_key == "RoadAnomaly":
                label[label == 2] = 1
            
            # Forward pass (once per image)
            logits = model(img_tensor)
            
            # Match resolution: logits <-> label
            if logits.shape[2] != label.shape[0] or logits.shape[3] != label.shape[1]:
                logits = F.interpolate(logits, size=label.shape[:2],
                                       mode='bilinear', align_corners=False)
            
            # Valid mask: only pixels 0 (in-dist) and 1 (anomaly), ignore 255
            valid_mask = (label == 0) | (label == 1)
            if valid_mask.sum() == 0:
                continue
            
            ood_gt = (label == 1).astype(np.int32)
            
            for method_name, score_fn in methods.items():
                score_map = score_fn(logits).squeeze().cpu().numpy()
                flat_scores = score_map[valid_mask].flatten()
                flat_labels = ood_gt[valid_mask].flatten()
                all_scores[method_name].append(flat_scores)
                all_labels[method_name].append(flat_labels)
            
            if (i + 1) % 20 == 0 or (i + 1) == len(paired):
                print(f"    [{i+1}/{len(paired)}]")
    
    # Compute metrics
    results = {}
    for method_name in methods:
        if not all_scores[method_name]:
            print(f"  {method_name}: no valid pixels!")
            results[method_name] = {"auprc": 0.0, "fpr95": 1.0}
            continue
        
        concat_scores = np.concatenate(all_scores[method_name])
        concat_labels = np.concatenate(all_labels[method_name])
        n_ood = (concat_labels == 1).sum()
        n_id  = (concat_labels == 0).sum()
        
        if n_ood == 0 or n_id == 0:
            print(f"  {method_name}: missing OOD ({n_ood}) or ID ({n_id}) pixels")
            results[method_name] = {"auprc": 0.0, "fpr95": 1.0}
            continue
        
        auprc = compute_auprc(concat_scores, concat_labels)
        fpr95 = compute_fpr_at_95_tpr(concat_scores, concat_labels)
        results[method_name] = {"auprc": auprc, "fpr95": fpr95}
    
    return results

print("Evaluation function defined")

## 9 — Run Evaluation on All Datasets

In [ ]:
# Force CPU to avoid CUDA compatibility issues
device = torch.device('cpu')
print(f'Device: {device}')

In [ ]:
model = model.cpu()
device = torch.device('cpu')

In [ ]:
# RUN EVALUATION

all_results = {}

for ds_key, ds_cfg in DATASETS.items():
    imgs = find_images(ds_cfg["image_dir"])
    if len(imgs) == 0:
        print(f"Skipping {ds_cfg['display_name']} - no images found")
        continue
    
    print(f'\n{"="*60}')
    print(f'Evaluating: {ds_cfg["display_name"]} ({ds_key})')
    print(f'{"="*60}')
    
    t0 = time.time()
    results = evaluate_dataset(ds_key, ds_cfg, model, METHODS, device)
    elapsed = time.time() - t0
    
    all_results[ds_key] = results
    # Save intermediate results to disk (survive disconnections)
    import json
    with open(os.path.join(RESULTS_DIR, "erfnet_partial_results.json"), "w") as f:
        json.dump({k: {m: {mk: float(mv) for mk, mv in met.items()} 
                       for m, met in v.items()} 
                   for k, v in all_results.items()}, f, indent=2)
    print(f"  Saved partial results to {RESULTS_DIR}")
    
    for method_name, metrics in results.items():
        print(f"  {method_name:12s} -> AuPRC: {metrics['auprc']*100:.2f}%  |  FPR@95: {metrics['fpr95']*100:.2f}%")
    print(f"  Time: {elapsed:.1f}s")

print("\nAll evaluations complete!")


## 10 — Results Table (Project Format)

In [ ]:
import pandas as pd

# Build the results table in the project format
rows = []
for method_name in METHODS:
    row = {"Model": "ERFNet", "Method": method_name}
    for ds_key, ds_cfg in DATASETS.items():
        display = ds_cfg["display_name"]
        if ds_key in all_results and method_name in all_results[ds_key]:
            m = all_results[ds_key][method_name]
            row[f"{display} AuPRC"] = f"{m['auprc']*100:.2f}"
            row[f"{display} FPR95"] = f"{m['fpr95']*100:.2f}"
        else:
            row[f"{display} AuPRC"] = "N/A"
            row[f"{display} FPR95"] = "N/A"
    rows.append(row)

df = pd.DataFrame(rows)
print("\n" + "="*100)
print("RESULTS TABLE — ERFNet Pixel-Based Anomaly Baselines")
print("="*100)
print(df.to_string(index=False))

# Also save as CSV
csv_path = os.path.join(RESULTS_DIR, "erfnet_anomaly_results.csv")
df.to_csv(csv_path, index=False)
print(f"\n💾 Results saved to: {csv_path}")


## 11 — Save Detailed Results

In [ ]:
import json

# Save full results as JSON
results_json = {}
for ds_key, ds_results in all_results.items():
    results_json[ds_key] = {}
    for method, metrics in ds_results.items():
        results_json[ds_key][method] = {
            "auprc": float(metrics["auprc"]),
            "fpr95": float(metrics["fpr95"]),
        }

json_path = os.path.join(RESULTS_DIR, "erfnet_results_detailed.json")
with open(json_path, "w") as f:
    json.dump(results_json, f, indent=2)

print(f"💾 Detailed results saved to: {json_path}")

# Also write a text summary
txt_path = os.path.join(RESULTS_DIR, "erfnet_results_summary.txt")
with open(txt_path, "w") as f:
    f.write("ERFNet Pixel-Based Anomaly Baselines\n")
    f.write("=" * 60 + "\n\n")
    for ds_key, ds_results in all_results.items():
        display = DATASETS[ds_key]["display_name"]
        f.write(f"{display}:\n")
        for method, metrics in ds_results.items():
            f.write(f"  {method:12s} → AuPRC: {metrics['auprc']*100:.2f}%  |  FPR@95: {metrics['fpr95']*100:.2f}%\n")
        f.write("\n")

print(f"💾 Summary saved to: {txt_path}")
print("\n✅ Done! Download results from /kaggle/working/results/erfnet/")
